# QAOA Vanilla - MaxCut on Weighted ER Graphs

This notebook:
- Generates 5 weighted Erdős–Rényi graphs
- Runs vanilla QAOA on each graph
- Collects expectation, variance, and optimal parameters

In [4]:
from qaoa import QAOA, problems, mixers, initialstates
from qiskit_algorithms.optimizers import L_BFGS_B
import time
from src.utils import generate_er_graphs, maxcut_bruteforce

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Configuration
n_graphs = 1
n_nodes = 10
depth = 10   # QAOA layers (p)

In [3]:
# Generate graphs (already weighted from your function)
graphs = generate_er_graphs(n_graphs=n_graphs, n_nodes=n_nodes)

print(f"Generated {len(graphs)} graphs")

Generated 1 graphs


In [4]:
# Inspect one graph
name, G = list(graphs.items())[0]
print("Example graph:", name)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Example graph: er_graph_0
Nodes: 10
Edges: 26


In [5]:

def graph_to_edgelist(G):
    """Convert NetworkX graph → [[u, v, w], ...]"""
    return [[u, v, d.get("weight", 1.0)] for u, v, d in G.edges(data=True)]


def run_qaoa_on_graph(graph_name, G, depth):
    print(f"\nRunning QAOA on {graph_name}")
    start = time.time()

    # --- Initialize QAOA
    qaoa = QAOA(
        problem=problems.MaxCut(G),
        mixer=mixers.X(),
        initialstate=initialstates.Plus(),
        interpolate=True,
        optimizer=[L_BFGS_B, {
            "maxiter": 50,
            "ftol": 1e-9
        }]
    )

    # --- Step 1: Landscape (optional, only meaningful for p=1)
    if depth == 1:
        qaoa.sample_cost_landscape()

    # --- Step 2: Optimization
    qaoa.optimize(depth=depth)

    # --- Step 3: Metrics
    exp_val = qaoa.get_Exp(depth=depth)
    var_val = qaoa.get_Var(depth=depth)

    gamma = qaoa.get_gamma(depth=depth)
    beta = qaoa.get_beta(depth=depth)

    # --- Step 4: Optimal (bruteforce)
    energy_opt, best_state = maxcut_bruteforce(G)

    # --- Step 5: Approximation ratio
    approx_ratio = exp_val / energy_opt if energy_opt != 0 else None

    # --- Step 6: Edge list
    edgelist_list = graph_to_edgelist(G)

    end = time.time()

    result = {
        "graph_name": graph_name,
        "n_nodes": G.number_of_nodes(),
        "edgelist_list_len": G.number_of_edges(),
        "n_layers": depth,
        "expected_energy": exp_val,
        "variance": var_val,
        "γ_coeff": gamma,
        "β_coeff": beta,
        "approx_ratio": approx_ratio,
        "energy_mqlib": energy_opt,
        "edgelist_list": edgelist_list,
        "took_time": round(end - start, 3),
        "method": "vanilla_qaoa",
        "optimizer": "BFGS"
    }

    print(f"Expectation: {exp_val}")
    print(f"Variance: {var_val}")

    return result

In [6]:
results = []

for graph_name, G in graphs.items():
    result = run_qaoa_on_graph(graph_name, G, depth)
    results.append(result)


Running QAOA on er_graph_0
2026-04-05 22:13:22 [info     ] Calculating energy landscape for depth p=1... file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 22:13:22 [info     ] Executing sample_cost_landscape file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 22:13:22 [info     ] circuits: 400                  file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 22:13:22 [info     ] Done execute                   file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 22:13:26 [info     ] Done measurement               file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 22:13:26 [info     ] Calculating Energy landscape done file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 22:13:28 [info     ] cost(depth 1 = -6.949082031250007 file=qaoa.qaoa func=optimize
2026-04-05 22:13:29 [info     ] cost(depth 2 = -6.971503906250003 file=qaoa.qaoa func=optimize
2026-04-05 22:13:33 [info     ] cost(depth 3 = -6.793164062500006 file=qaoa.qaoa func=optimize
2026-04-05 22:13:37 [info     ] cost(d

In [7]:
# for x in dir(qaoa):
#     if not x.startswith("_"):    
#         print(x)

In [8]:
# Convert to DataFrame
df = pd.DataFrame(results)

df

,graph_name,n_nodes,edgelist_list_len,n_layers,expected_energy,variance,γ_coeff,β_coeff,approx_ratio,energy_mqlib,edgelist_list,took_time,method,optimizer
0,er_graph_0,10,26,10,-7.486533,1.156016,"[0.35069707260152094, 0.37071074229956863, 0.27603118966246903, 0.2286274656324677, 0.29371319556948594, 0.3481232929618466, 0.3394966899920736, 0.33048619417575675, 0.30573873905097976, 0.2629422393734665]","[0.17097588726181231, 0.019404455595772068, 0.11549086047562834, 0.14753367926999852, 0.0954500294385663, 0.031031059435742688, 0.008292737944830032, 0.04276962111190659, 0.1281593308163348, 0.18824054019491188]",0.858547,-8.72,"[[0, 5, 0.23], [0, 6, 0.01], [0, 7, 0.7], [1, 2, 0.04], [1, 3, 0.77], [1, 4, 0.46], [1, 7, 0.24], [1, 8, 0.12], [1, 9, 0.49], [2, 3, 0.44], [2, 4, 0.09], [2, 6, 0.4], [2, 8, 0.41], [2, 9, 0.82], [3, 4, 1.0], [3, 5, 0.85], [3, 6, 0.64], [3, 7, 0.36], [3, 8, 0.51], [4, 6, 0.79], [4, 7, 0.23], [4, 8, 0.37], [5, 6, 0.24], [7, 8, 0.15], [7, 9, 0.38], [8, 9, 0.58]]",74.904,vanilla_qaoa,BFGS


In [ ]:
# Sort by best expectation (MaxCut → usually minimize energy)
df_sorted = df.sort_values(by="expected_energy")

df_sorted

KeyError: 'expectation'

In [ ]:
# Done
print("QAOA experiment complete ✅")

QAOA experiment complete ✅


In [5]:
df = pd.read_csv("ADAPT.jl_results/test/9_nodes/qaoa_result/qaoa.csv")

In [6]:
df

,graph_name,n_nodes,edgelist_list_len,n_layers,expected_energy,variance,γ_coeff,β_coeff,approx_ratio,energy_mqlib,edgelist_list,took_time,method,optimizer,run_id
0,graph_1,9,27,9,-9.287578,1.178839,[0.31434736 0.35153371 0.38302948 0.41910144 0.45197442 0.48984374\n 0.5171514 0.55883106 0.59080838],[4.92125412 4.92002712 4.92419603 4.92826849 4.93595683 4.93377345\n 4.94047746 4.94410176 4.95039262],0.90699,-10.24,"[[1, 2, 0.48], [1, 3, 0.21], [1, 4, 0.64], [1, 5, 0.48], [1, 6, 0.51], [2, 3, 0.19], [2, 4, 0.81], [2, 5, 0.01], [2, 6, 0.94], [2, 8, 0.18], [2, 9, 0.24], [3, 4, 0.49], [3, 5, 0.45], [3, 6, 0.27], [3, 7, 0.97], [3, 8, 0.22], [4, 6, 0.39], [4, 7, 0.97], [4, 8, 0.18], [4, 9, 0.43], [5, 6, 0.02], [5, 7, 0.93], [5, 8, 0.42], [5, 9, 0.88], [6, 7, 0.92], [6, 8, 0.53], [7, 9, 0.99]]",84.284,vanilla_qaoa,BFGS,0
